# WorldSim Parallel Slot Benchmark
Measuring `-np 1` vs `-np 2` vs `-np 4` throughput for Qwen3.5 0.8B/2B


## 1. Setup & Path Discovery


In [ ]:
from pathlib import Path
import json
import os
import signal
import statistics
import subprocess
import sys
import time

cwd = Path.cwd()
repo_marker = cwd / 'training/run_qlora_train.py'
notebook_marker = cwd / 'notebooks/dgx_spark_parallel_benchmark.ipynb'
assert repo_marker.exists() and notebook_marker.exists(), (
    'Run this notebook from the repository root: worldsim-training'
)
REPO_ROOT = cwd
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

LLAMA_SERVER = REPO_ROOT / 'tools' / 'llama.cpp' / 'build' / 'bin' / 'llama-server'
GGUF_DIR = REPO_ROOT / 'artifacts' / 'gguf'
MODEL_08B = GGUF_DIR / 'worldsim-v2-qwen3.5-0.8b-q4_k_m.gguf'
MODEL_2B = GGUF_DIR / 'worldsim-v2-qwen3.5-2b-q4_k_m.gguf'
HAS_2B = MODEL_2B.exists()

assert LLAMA_SERVER.exists(), f'llama-server not found: {LLAMA_SERVER}'
assert MODEL_08B.exists(), f'0.8B GGUF not found: {MODEL_08B}'

subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        'aiohttp',
        '--quiet',
        '--break-system-packages',
    ],
    capture_output=True,
    check=False,
)

print(f'llama-server: {LLAMA_SERVER}')
print(f'0.8B GGUF:    {MODEL_08B} ({MODEL_08B.stat().st_size / 1e6:.0f} MB)')
print(f'2B GGUF:      {'✅ ' + str(MODEL_2B) if HAS_2B else '❌ not found (skip 2B tests)'}')
print('Setup complete ✅')


## 2. Server Management & Request Helpers


In [ ]:
import asyncio
import json
import statistics
import subprocess
import time
import urllib.request
from dataclasses import dataclass
from typing import Optional

import aiohttp

SERVER_PORT = 8090
SERVER_URL = f'http://127.0.0.1:{SERVER_PORT}'


def start_server(model_path: Path, n_parallel: int, ctx_size: int = 2048, n_gpu_layers: int = 99, extra_args: list[str] | None = None):
    """Start llama-server and wait until healthy."""
    args = [
        str(LLAMA_SERVER),
        '-m', str(model_path),
        '--host', '127.0.0.1',
        '--port', str(SERVER_PORT),
        '-c', str(ctx_size),
        '-np', str(n_parallel),
        '-ngl', str(n_gpu_layers),
        '-fa', 'on',
        '--jinja',
        '--no-webui',
        '--metrics',
        '--chat-template-kwargs', '{"enable_thinking": false}',
        '-ctk', 'q8_0',
        '-ctv', 'q8_0',
    ]
    if extra_args:
        args.extend(extra_args)

    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    for attempt in range(120):
        time.sleep(0.5)
        try:
            resp = urllib.request.urlopen(f'{SERVER_URL}/health', timeout=2)
            data = json.loads(resp.read())
            if data.get('status') == 'ok':
                print(f'Server ready ✅ (attempt {attempt + 1}, pid={proc.pid}, np={n_parallel})')
                return proc
        except Exception:
            pass
        if proc.poll() is not None:
            stderr = proc.stderr.read().decode()[-500:]
            raise RuntimeError(f'Server died during startup: {stderr}')

    proc.kill()
    raise RuntimeError('Server failed to become healthy in 60s')


def stop_server(proc) -> None:
    """Gracefully stop server."""
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
    print('Server stopped 🛑')


@dataclass
class RequestResult:
    request_id: int
    task_type: str
    success: bool
    latency_ms: float
    tokens_generated: int
    prompt_tokens: int
    error: Optional[str] = None
    json_valid: bool = False


async def send_request(session: aiohttp.ClientSession, request_id: int, task_type: str,
                       messages: list, max_tokens: int = 64, temperature: float = 0,
                       response_format: dict | None = None) -> RequestResult:
    """Send a single chat completion request and measure latency."""
    payload = {
        'messages': messages,
        'max_tokens': max_tokens,
        'temperature': temperature,
        'stream': False,
    }
    if response_format:
        payload['response_format'] = response_format

    start = time.perf_counter()
    try:
        async with session.post(
            f'{SERVER_URL}/v1/chat/completions',
            json=payload,
            timeout=aiohttp.ClientTimeout(total=30),
        ) as resp:
            elapsed_ms = (time.perf_counter() - start) * 1000
            data = await resp.json()

            if resp.status != 200:
                return RequestResult(request_id, task_type, False, elapsed_ms, 0, 0, error=str(data.get('error', 'unknown')))

            choice = data['choices'][0]
            content = choice['message']['content']
            usage = data.get('usage', {})

            json_valid = False
            try:
                json.loads(content)
                json_valid = True
            except (json.JSONDecodeError, TypeError):
                pass

            return RequestResult(
                request_id=request_id,
                task_type=task_type,
                success=True,
                latency_ms=elapsed_ms,
                tokens_generated=usage.get('completion_tokens', 0),
                prompt_tokens=usage.get('prompt_tokens', 0),
                json_valid=json_valid,
            )
    except Exception as exc:
        elapsed_ms = (time.perf_counter() - start) * 1000
        return RequestResult(request_id, task_type, False, elapsed_ms, 0, 0, error=str(exc))


async def run_concurrent_requests(requests: list[dict], concurrency: int) -> list[RequestResult]:
    """Send multiple requests with controlled concurrency."""
    semaphore = asyncio.Semaphore(concurrency)

    async def bounded_request(session: aiohttp.ClientSession, req: dict) -> RequestResult:
        async with semaphore:
            return await send_request(session, **req)

    async with aiohttp.ClientSession() as session:
        tasks = [bounded_request(session, req) for req in requests]
        results = await asyncio.gather(*tasks)

    return list(results)


def _percentile(values: list[float], ratio: float) -> int:
    if not values:
        return 0
    ordered = sorted(values)
    index = min(max(int(len(ordered) * ratio), 0), len(ordered) - 1)
    return round(ordered[index])


def compute_stats(results: list[RequestResult]) -> dict:
    """Compute latency statistics from results."""
    successful = [r for r in results if r.success]
    latencies = [r.latency_ms for r in successful]

    if not latencies:
        return {'count': len(results), 'success': 0, 'fail': len(results), 'json_valid': 0}

    return {
        'count': len(results),
        'success': len(successful),
        'fail': len(results) - len(successful),
        'json_valid': sum(1 for r in successful if r.json_valid),
        'latency_p50': _percentile(latencies, 0.50),
        'latency_p95': _percentile(latencies, 0.95),
        'latency_p99': _percentile(latencies, 0.99),
        'latency_mean': round(statistics.mean(latencies)),
        'latency_min': round(min(latencies)),
        'latency_max': round(max(latencies)),
        'total_time_ms': round(max(r.latency_ms for r in results)),
        'avg_gen_tokens': round(statistics.mean(r.tokens_generated for r in successful)),
        'avg_prompt_tokens': round(statistics.mean(r.prompt_tokens for r in successful)),
        'throughput_req_per_min': round(len(successful) / (max(latencies) / 60000), 1) if latencies else 0,
    }


print('Helpers loaded ✅')


## 3. Test Prompts — WorldSim Task Types


In [ ]:
TASK_E_SCHEMA = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'task_e',
        'strict': True,
        'schema': {
            'type': 'object',
            'properties': {
                'action_id': {'type': 'integer', 'enum': [0, 1, 2, 3]},
                'confidence': {'type': 'number'},
                'hint_en': {'type': 'string'},
            },
            'required': ['action_id', 'confidence', 'hint_en'],
            'additionalProperties': False,
        },
    },
}

TASK_M_SCHEMA = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'task_m',
        'strict': True,
        'schema': {
            'type': 'object',
            'properties': {
                'decision_id': {'type': 'integer', 'enum': [0, 1, 2, 3]},
                'confidence': {'type': 'number'},
                'dissent_risk': {'type': 'number'},
                'hint_en': {'type': 'string'},
            },
            'required': ['decision_id', 'confidence', 'dissent_risk', 'hint_en'],
            'additionalProperties': False,
        },
    },
}

TASK_E_MSG = [
    {'role': 'system', 'content': 'You are a Stone Age agent. Output JSON only.'},
    {'role': 'user', 'content': '''[TASK] E
[TEMP]
NS=0.8 HA=0.2 RD=0.5 P=0.7 type=choleric
[personality]
bold, impulsive
[situation]
stranger group approaching
[options]
0:flee 1:hide 2:confront 3:warn
[output]
{"action_id": number, "confidence": 0.0-1.0, "hint_en": "one sentence"}'''},
]

TASK_M_MSG = [
    {'role': 'system', 'content': 'You are a tribal council. Output JSON only.'},
    {'role': 'user', 'content': '''[TASK] M
[group]
mean_H=0.4 var_H=0.3 leader=aggressive
[situation]
neighbor offers alliance after past conflict
[options]
0:accept 1:reject 2:counter_terms 3:delay
[output]
{"decision_id": number, "confidence": 0.0-1.0, "dissent_risk": 0.0-1.0, "hint_en": "reason"}'''},
]

TASK_B_MSG = [
    {'role': 'system', 'content': '당신은 WorldSim 서술자입니다. 해라체로 2-3문장 서술하세요.'},
    {'role': 'user', 'content': '''[TASK] B
[TEMP]
NS=0.2 HA=0.8 RD=0.6 P=0.4 type=melancholic
[personality]
cautious, anxious
[emotion]
fear:0.8
[situation]
predator spotted near camp'''},
]


def make_requests(n: int, task_type: str = 'E') -> list[dict]:
    """Generate n request dicts for a given task type."""
    requests: list[dict] = []
    for i in range(n):
        if task_type == 'E':
            requests.append({
                'request_id': i,
                'task_type': 'E',
                'messages': TASK_E_MSG,
                'max_tokens': 64,
                'temperature': 0,
                'response_format': TASK_E_SCHEMA,
            })
        elif task_type == 'M':
            requests.append({
                'request_id': i,
                'task_type': 'M',
                'messages': TASK_M_MSG,
                'max_tokens': 128,
                'temperature': 0.3,
                'response_format': TASK_M_SCHEMA,
            })
        elif task_type == 'B':
            requests.append({
                'request_id': i,
                'task_type': 'B',
                'messages': TASK_B_MSG,
                'max_tokens': 150,
                'temperature': 0.7,
            })
        elif task_type == 'mixed':
            cycle = ['E', 'M', 'B']
            chosen = cycle[i % 3]
            mixed_req = make_requests(1, chosen)[0]
            mixed_req['request_id'] = i
            requests.append(mixed_req)
        else:
            raise ValueError(f'Unsupported task type: {task_type}')
    return requests


print(f'Task E prompt: ~{len(json.dumps(TASK_E_MSG))} chars')
print(f'Task M prompt: ~{len(json.dumps(TASK_M_MSG))} chars')
print(f'Task B prompt: ~{len(json.dumps(TASK_B_MSG))} chars')
print('Test prompts ready ✅')


## 4. Experiment 1: Sequential vs Parallel Slots (0.8B)


In [ ]:
N_REQUESTS = 12
RESULTS = {}

for np_slots in [1, 2, 4]:
    print()
    print('=' * 60)
    print(f'  EXPERIMENT: 0.8B, np={np_slots}, {N_REQUESTS} requests (Task E)')
    print('=' * 60)

    ctx_per_slot = 2048 // np_slots
    proc = start_server(MODEL_08B, n_parallel=np_slots, ctx_size=2048)

    try:
        warmup = asyncio.get_event_loop().run_until_complete(
            run_concurrent_requests(make_requests(1, 'E'), concurrency=1)
        )
        print(f'  Warmup: {warmup[0].latency_ms:.0f}ms')

        wall_start = time.perf_counter()
        results = asyncio.get_event_loop().run_until_complete(
            run_concurrent_requests(make_requests(N_REQUESTS, 'E'), concurrency=np_slots)
        )
        wall_elapsed = (time.perf_counter() - wall_start) * 1000

        stats = compute_stats(results)
        stats['np_slots'] = np_slots
        stats['ctx_per_slot'] = ctx_per_slot
        stats['wall_clock_ms'] = round(wall_elapsed)
        stats['effective_throughput'] = round(stats['success'] / (wall_elapsed / 1000), 2)

        RESULTS[f'08b_np{np_slots}_taskE'] = stats

        print(f"  Results: {stats['success']}/{stats['count']} success")
        print(f"  Latency P50: {stats['latency_p50']}ms  P95: {stats['latency_p95']}ms")
        print(f'  Wall clock: {wall_elapsed:.0f}ms')
        print(f"  Throughput: {stats['effective_throughput']} req/s")
    finally:
        stop_server(proc)
        time.sleep(2)

print()
print('Experiment 1 complete ✅')


## 5. Experiment 2: Mixed Task Types in Parallel


In [ ]:
for np_slots in [1, 2]:
    print()
    print('=' * 60)
    print(f'  EXPERIMENT: 0.8B, np={np_slots}, {N_REQUESTS} mixed requests (E+M+B)')
    print('=' * 60)

    proc = start_server(MODEL_08B, n_parallel=np_slots, ctx_size=2048)

    try:
        asyncio.get_event_loop().run_until_complete(
            run_concurrent_requests(make_requests(1, 'E'), concurrency=1)
        )

        wall_start = time.perf_counter()
        results = asyncio.get_event_loop().run_until_complete(
            run_concurrent_requests(make_requests(N_REQUESTS, 'mixed'), concurrency=np_slots)
        )
        wall_elapsed = (time.perf_counter() - wall_start) * 1000

        for task_type in ['E', 'M', 'B']:
            task_results = [r for r in results if r.task_type == task_type]
            if task_results:
                task_stats = compute_stats(task_results)
                print(f"  Task {task_type}: {task_stats['success']}/{task_stats['count']} success, P50={task_stats['latency_p50']}ms")

        stats = compute_stats(results)
        stats['np_slots'] = np_slots
        stats['wall_clock_ms'] = round(wall_elapsed)
        stats['effective_throughput'] = round(stats['success'] / (wall_elapsed / 1000), 2)
        RESULTS[f'08b_np{np_slots}_mixed'] = stats

        print(f"  Total: {stats['success']}/{stats['count']}, Wall: {wall_elapsed:.0f}ms, Throughput: {stats['effective_throughput']} req/s")
    finally:
        stop_server(proc)
        time.sleep(2)

print()
print('Experiment 2 complete ✅')


## 6. Experiment 3: Free Text vs Grammar-Constrained


In [ ]:
for mode, task_type in [('grammar', 'E'), ('free_text', 'B')]:
    print()
    print('=' * 60)
    print(f'  EXPERIMENT: 0.8B, np=2, {mode} ({task_type})')
    print('=' * 60)

    proc = start_server(MODEL_08B, n_parallel=2, ctx_size=2048)

    try:
        asyncio.get_event_loop().run_until_complete(
            run_concurrent_requests(make_requests(1, task_type), concurrency=1)
        )

        wall_start = time.perf_counter()
        results = asyncio.get_event_loop().run_until_complete(
            run_concurrent_requests(make_requests(N_REQUESTS, task_type), concurrency=2)
        )
        wall_elapsed = (time.perf_counter() - wall_start) * 1000

        stats = compute_stats(results)
        stats['mode'] = mode
        stats['wall_clock_ms'] = round(wall_elapsed)
        stats['effective_throughput'] = round(stats['success'] / (wall_elapsed / 1000), 2)
        RESULTS[f'08b_np2_{mode}'] = stats

        print(f"  {stats['success']}/{stats['count']} success, P50={stats['latency_p50']}ms, Throughput={stats['effective_throughput']} req/s")
    finally:
        stop_server(proc)
        time.sleep(2)

print()
print('Experiment 3 complete ✅')


## 7. Experiment 4: 2B Model (if available)


In [ ]:
if HAS_2B:
    for np_slots in [1, 2]:
        print()
        print('=' * 60)
        print(f'  EXPERIMENT: 2B, np={np_slots}, {N_REQUESTS} requests (Task E)')
        print('=' * 60)

        proc = start_server(MODEL_2B, n_parallel=np_slots, ctx_size=2048)

        try:
            asyncio.get_event_loop().run_until_complete(
                run_concurrent_requests(make_requests(1, 'E'), concurrency=1)
            )

            wall_start = time.perf_counter()
            results = asyncio.get_event_loop().run_until_complete(
                run_concurrent_requests(make_requests(N_REQUESTS, 'E'), concurrency=np_slots)
            )
            wall_elapsed = (time.perf_counter() - wall_start) * 1000

            stats = compute_stats(results)
            stats['np_slots'] = np_slots
            stats['wall_clock_ms'] = round(wall_elapsed)
            stats['effective_throughput'] = round(stats['success'] / (wall_elapsed / 1000), 2)
            RESULTS[f'2b_np{np_slots}_taskE'] = stats

            print(f"  {stats['success']}/{stats['count']} success, P50={stats['latency_p50']}ms")
            print(f"  Wall: {wall_elapsed:.0f}ms, Throughput: {stats['effective_throughput']} req/s")
        finally:
            stop_server(proc)
            time.sleep(2)
else:
    print('2B model not available, skipping ⏭️')

print()
print('Experiment 4 complete ✅')


## 8. Results Summary


In [ ]:
print('=' * 80)
print('  PARALLEL SLOT BENCHMARK — FULL RESULTS')
print('=' * 80)

print()
print(f"{'Config':<30} {'Success':>8} {'P50(ms)':>8} {'P95(ms)':>8} {'Wall(ms)':>9} {'Throughput':>11}")
print('-' * 80)

for key, stats in sorted(RESULTS.items()):
    label = key.replace('08b_', '0.8B ').replace('2b_', '2B ').replace('_', ' ')
    print(f"{label:<30} {stats['success']:>5}/{stats['count']:<2} {stats.get('latency_p50', '—'):>7} {stats.get('latency_p95', '—'):>7} {stats.get('wall_clock_ms', '—'):>8} {stats.get('effective_throughput', '—'):>9} req/s")

print()
print('=' * 80)
print('  SPEEDUP ANALYSIS')
print('=' * 80)

baseline_key = '08b_np1_taskE'
if baseline_key in RESULTS:
    base_wall = RESULTS[baseline_key]['wall_clock_ms']
    for key, stats in sorted(RESULTS.items()):
        if 'taskE' in key and key != baseline_key:
            speedup = base_wall / stats['wall_clock_ms'] if stats['wall_clock_ms'] > 0 else 0
            label = key.replace('08b_', '0.8B ').replace('2b_', '2B ').replace('_', ' ')
            print(f'  {label}: {speedup:.2f}x vs baseline (np=1)')

results_path = REPO_ROOT / 'outputs' / 'benchmarks' / 'parallel_slot_benchmark.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
with open(results_path, 'w', encoding='utf-8') as handle:
    json.dump(RESULTS, handle, indent=2, ensure_ascii=False)
print()
print(f'Full results saved to: {results_path} 📁')


## 9. Recommendation


In [ ]:
print('=' * 60)
print('  RECOMMENDATION FOR WORLDSIM')
print('=' * 60)

np1 = RESULTS.get('08b_np1_taskE', {})
np2 = RESULTS.get('08b_np2_taskE', {})
np4 = RESULTS.get('08b_np4_taskE', {})

wall_speedup = 0.0
rec_08b = '?'

if np1 and np2:
    wall_speedup = np1.get('wall_clock_ms', 1) / max(np2.get('wall_clock_ms', 1), 1)
    p50_ratio = np2.get('latency_p50', 1) / max(np1.get('latency_p50', 1), 1)

    print()
    print('  0.8B np=2 vs np=1:')
    print(f'    Wall-clock speedup: {wall_speedup:.2f}x')
    print(f'    P50 latency ratio:  {p50_ratio:.2f}x (>1 means slower per request)')

    if wall_speedup > 1.3 and p50_ratio < 1.5:
        rec_08b = 2
        print('    RECOMMEND: np=2 for 0.8B ✅')
    elif wall_speedup > 1.1:
        rec_08b = 2
        print('    MARGINAL: np=2 is slightly better ⚠️')
    else:
        rec_08b = 1
        print('    NO BENEFIT: keep np=1 for 0.8B ❌')

if np1 and np4:
    wall_speedup_4 = np1.get('wall_clock_ms', 1) / max(np4.get('wall_clock_ms', 1), 1)
    print()
    print(f'  0.8B np=4 vs np=1: {wall_speedup_4:.2f}x wall-clock speedup')
    if wall_speedup and wall_speedup_4 < 1.3 * wall_speedup:
        print('    DIMINISHING RETURNS: np=4 not worth the RAM/context cost ❌')

print()
print('  Final recommendation:')
print(f'    0.8B: np={rec_08b}')
print('    2B:   np=1 (complex tasks need full context)')
print()
print('=' * 60)
